In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support
from sklearn.datasets import load_breast_cancer
import numpy as np
import pandas as pd

from src.evaluation.sklearn_evaluator import SklearnEvaluator

In [2]:
def preprocessing_fn(data, train_index, test_index):
    data_train = data.iloc[train_index].copy()
    data_train_means = data_train.mean()
    data_train_stds = data_train.std()
    data_train = (data_train - data_train_means) / data_train_stds

    data_test = data.iloc[test_index].copy()
    data_test = (data_test - data_train_means) / data_train_stds
    return data_train, data_test

In [3]:
precision_recall_f2score_support = lambda *args, **kwargs: precision_recall_fscore_support(*args, beta=2., **kwargs)

In [4]:
toy_df = load_breast_cancer(as_frame=True)

In [5]:
model = LogisticRegression()

evaluator = SklearnEvaluator(
    model=model,
    preprocessing_fn=preprocessing_fn,
    inference_fn=model.predict,
    X=toy_df['data'],
    Y=toy_df['frame'][['target']],
    data_groups=toy_df['data'].index,
    score_fns=[precision_recall_f2score_support]
)

results = evaluator.cross_validation(n_folds=5, iterations=5)

Cross validation:   0%|          | 0/5 [00:00<?, ?it/s]

Splits evaluation:   0%|          | 0/5 [00:00<?, ?it/s]

Splits evaluation:   0%|          | 0/5 [00:00<?, ?it/s]

Splits evaluation:   0%|          | 0/5 [00:00<?, ?it/s]

Splits evaluation:   0%|          | 0/5 [00:00<?, ?it/s]

Splits evaluation:   0%|          | 0/5 [00:00<?, ?it/s]

In [6]:
for class_label in [0, 1]:
    mean_values = np.array(results['<lambda>'])[:, :, :, class_label].squeeze().mean(0)
    std_values = np.array(results['<lambda>'])[:, :, :, class_label].squeeze().std(0)

    metrics_df = pd.DataFrame({
        'Metric': ['Precision', 'Recall', 'F2', 'Support'],
        'Mean': mean_values,
        'Std': std_values
    })
    metrics_df.set_index('Metric', inplace=True)
    print(metrics_df)
    print(f'Label: "{class_label}"', end='\n\n')

                Mean       Std
Metric                        
Precision   0.980372  0.020896
Recall      0.962581  0.029979
F2          0.965781  0.022820
Support    42.400000  4.766550
Label: "0"

                Mean       Std
Metric                        
Precision   0.977934  0.017771
Recall      0.989026  0.011621
F2          0.986682  0.008276
Support    71.400000  4.866210
Label: "1"

